In [1]:
# ── MLP Autoencoder ───────────────────────────────────────────
import random
import tensorflow as tf
import gc
import time
import numpy as np
import os
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Flatten, Reshape
from tensorflow.keras.layers import Conv1D, GRU, LSTM, Bidirectional
from tensorflow.keras.layers import TimeDistributed, RepeatVector
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# 시드 고정
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

print("설정 완료")

2026-04-07 23:17:57.687985: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775603878.029393      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775603878.131734      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775603878.862626      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775603878.862685      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775603878.862688      17 computation_placer.cc:177] computation placer alr

설정 완료


In [2]:
# 데이터 로드
PREP = 'PREP_DIR'
X_train = np.load(PREP + 'X_train_w5.npy')
X_val   = np.load(PREP + 'X_val_w5.npy')
X_te_tue = np.load(PREP + 'X_te_tue_w5.npy')
X_te_wed = np.load(PREP + 'X_te_wed_w5.npy')
X_te_fri = np.load(PREP + 'X_te_fri_w5.npy')
y_te_tue = np.load(PREP + 'y_te_tue_w5.npy', allow_pickle=True)
y_te_wed = np.load(PREP + 'y_te_wed_w5.npy', allow_pickle=True)
y_te_fri = np.load(PREP + 'y_te_fri_w5.npy', allow_pickle=True)

TIMESTEPS = X_train.shape[1]
FEATURES  = X_train.shape[2]
print(f"Train: {X_train.shape} / Val: {X_val.shape}")

Train: (238264, 5, 78) / Val: (26474, 5, 78)


In [3]:
# 공통 함수
def get_threshold(model, X_val, percentile=95):
    X_pred = model.predict(X_val, verbose=0)
    mse = np.mean(np.square(X_val - X_pred), axis=(1, 2))
    return np.percentile(mse, percentile)

def get_mse(model, X):
    X_pred = model.predict(X, verbose=0)
    return np.mean(np.square(X - X_pred), axis=(1, 2))

def evaluate(model, threshold, name):
    results = {}
    for X_te, y_te, attack in [
        (X_te_tue, y_te_tue, 'BruteForce'),
        (X_te_wed, y_te_wed, 'DoS'),
        (X_te_fri, y_te_fri, 'DDoS'),
    ]:
        mse    = get_mse(model, X_te)
        y_pred = (mse > threshold).astype(int)
        y_true = (y_te != 'BENIGN').astype(int)
        report = classification_report(y_true, y_pred, output_dict=True)
        recall    = report['1']['recall']    if '1' in report else 0
        precision = report['1']['precision'] if '1' in report else 0
        f1        = report['1']['f1-score']  if '1' in report else 0
        results[attack] = {'recall': recall, 'precision': precision, 'f1': f1}
        print(f"  [{name}] {attack} → Recall: {recall:.3f}  Precision: {precision:.3f}  F1: {f1:.3f}")
    return results

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("\n========== 함수 완료 ==========")



========== 함수 완료 ==========


In [4]:
# MLP 모델
print("\n========== EXP1: MLP ==========")
inp = Input(shape=(TIMESTEPS, FEATURES))
x   = Flatten()(inp)
x   = Dense(128, activation='relu')(x)
x   = Dense(64,  activation='relu')(x)
x   = Dense(128, activation='relu')(x)
out = Dense(TIMESTEPS * FEATURES, activation='sigmoid')(x)
out = Reshape((TIMESTEPS, FEATURES))(out)
mlp_model = Model(inp, out)
mlp_model.compile(optimizer='adam', loss='mse')
mlp_model.summary()

start = time.time()
mlp_hist = mlp_model.fit(
    X_train, X_train,
    epochs=100, batch_size=256,
    validation_data=(X_val, X_val),
    callbacks=[es], shuffle=True, verbose=1
)
mlp_time = (time.time() - start) / len(mlp_hist.history['loss'])
mlp_threshold = get_threshold(mlp_model, X_val)
print(f"MLP 임계값: {mlp_threshold:.6f} / 에폭당 평균: {mlp_time:.2f}s")
mlp_results = evaluate(mlp_model, mlp_threshold, 'MLP')

os.makedirs('OUTPUT_DIRmodels', exist_ok=True)
mlp_model.save('OUTPUT_DIRmodels/mlp_model.keras')
np.save('OUTPUT_DIRmodels/mlp_threshold.npy', mlp_threshold)

# 메모리 해제
del mlp_model
gc.collect()
print("MLP 완료 + 저장!")


========== EXP1: MLP ==========


2026-04-07 23:19:01.153764: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 5, 78)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 390)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        50,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 390)            │        50,310 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 5, 78)          │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 116,934 (456.77 KB)

 Trainable params: 116,934 (456.77 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - loss: 0.0289 - val_loss: 0.0039
Epoch 2/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 0.0035 - val_loss: 0.0026
Epoch 3/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0024 - val_loss: 0.0021
Epoch 4/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0021 - val_loss: 0.0019
Epoch 5/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 0.0019 - val_loss: 0.0017
Epoch 6/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 0.0017 - val_loss: 0.0016
Epoch 7/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 0.0015 - val_loss: 0.0014
Epoch 8/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 0.0014 - val_loss: 0.0011
Epoch 9/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 0.0010 - val_loss: 8.4812e-04
Epoch 10/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 8.0383e-04 - val_loss: 5.5891e-04
Epoch 11/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 7s 7ms/step - loss: 4.9902e-04 - val_loss: 3.0400e-04
Epoch 12/100
931/931 

In [5]:
# ── 1D-CNN Autoencoder ────────────────────────────────────────
import gc
from tensorflow.keras.layers import Conv1D, TimeDistributed

print("\n========== EXP2: 1D-CNN ==========")
inp = Input(shape=(TIMESTEPS, FEATURES))
x   = Conv1D(64, kernel_size=3, activation='relu', padding='same')(inp)
x   = Conv1D(32, kernel_size=3, activation='relu', padding='same')(x)
x   = Conv1D(64, kernel_size=3, activation='relu', padding='same')(x)
out = TimeDistributed(Dense(FEATURES))(x)
cnn_model = Model(inp, out)
cnn_model.compile(optimizer='adam', loss='mse')
cnn_model.summary()

start = time.time()
cnn_hist = cnn_model.fit(
    X_train, X_train,
    epochs=100, batch_size=256,
    validation_data=(X_val, X_val),
    callbacks=[es], shuffle=True, verbose=1
)
cnn_time = (time.time() - start) / len(cnn_hist.history['loss'])
cnn_threshold = get_threshold(cnn_model, X_val)
print(f"CNN 임계값: {cnn_threshold:.6f} / 에폭당 평균: {cnn_time:.2f}s")
cnn_results = evaluate(cnn_model, cnn_threshold, '1D-CNN')

cnn_model.save('OUTPUT_DIRmodels/cnn_model.keras')
np.save('OUTPUT_DIRmodels/cnn_threshold.npy', cnn_threshold)

del cnn_model
gc.collect()
print("CNN 완료 + 저장!")


========== EXP2: 1D-CNN ==========


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 5, 78)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 5, 64)          │        15,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 5, 32)          │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 5, 64)          │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 5, 78)          │         5,070 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,494 (126.93 KB)

 Trainable params: 32,494 (126.93 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - loss: 0.0100 - val_loss: 2.4642e-04
Epoch 2/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 2.0763e-04 - val_loss: 1.6241e-04
Epoch 3/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 1.2429e-04 - val_loss: 9.7205e-05
Epoch 4/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 8.5034e-05 - val_loss: 8.3857e-05
Epoch 5/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 10s 10ms/step - loss: 6.6514e-05 - val_loss: 5.8054e-05
Epoch 6/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 11s 12ms/step - loss: 5.6279e-05 - val_loss: 5.1296e-05
Epoch 7/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 4.9651e-05 - val_loss: 4.5264e-05
Epoch 8/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 4.4371e-05 - val_loss: 4.6567e-05
Epoch 9/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - loss: 4.0386e-05 - val_loss: 3.5165e-05
Epoch 10/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - loss: 3.6585e-05 - val_loss: 5.8172e-05
Epoch 11/100
931/931 ━━━━━━━━━━━━

In [6]:
# ── GRU Autoencoder ───────────────────────────────────────────
from tensorflow.keras.layers import GRU, RepeatVector

print("\n========== EXP3: GRU ==========")
inp     = Input(shape=(TIMESTEPS, FEATURES))
encoded = GRU(64, activation='relu', return_sequences=False)(inp)
repeated= RepeatVector(TIMESTEPS)(encoded)
decoded = GRU(64, activation='relu', return_sequences=True)(repeated)
out     = TimeDistributed(Dense(FEATURES))(decoded)
gru_model = Model(inp, out)
gru_model.compile(optimizer='adam', loss='mse')
gru_model.summary()

start = time.time()
gru_hist = gru_model.fit(
    X_train, X_train,
    epochs=100, batch_size=256,
    validation_data=(X_val, X_val),
    callbacks=[es], shuffle=True, verbose=1
)
gru_time = (time.time() - start) / len(gru_hist.history['loss'])
gru_threshold = get_threshold(gru_model, X_val)
print(f"GRU 임계값: {gru_threshold:.6f} / 에폭당 평균: {gru_time:.2f}s")
gru_results = evaluate(gru_model, gru_threshold, 'GRU')

gru_model.save('OUTPUT_DIRmodels/gru_model.keras')
np.save('OUTPUT_DIRmodels/gru_threshold.npy', gru_threshold)

del gru_model
gc.collect()
print("GRU 완료 + 저장!")


========== EXP3: GRU ==========


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 5, 78)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 64)             │        27,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector (RepeatVector)    │ (None, 5, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 5, 64)          │        24,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 5, 78)          │         5,070 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57,678 (225.30 KB)

 Trainable params: 57,678 (225.30 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 25s 22ms/step - loss: 0.0173 - val_loss: 0.0057
Epoch 2/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - loss: 0.0050 - val_loss: 0.0032
Epoch 3/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 21s 22ms/step - loss: 0.0030 - val_loss: 0.0027
Epoch 4/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - loss: 0.0026 - val_loss: 0.0023
Epoch 5/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - loss: 0.0023 - val_loss: 0.0021
Epoch 6/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - loss: 0.0021 - val_loss: 0.0020
Epoch 7/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - loss: 0.0020 - val_loss: 0.0019
Epoch 8/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - loss: 0.0018 - val_loss: 0.0017
Epoch 9/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - loss: 0.0017 - val_loss: 0.0017
Epoch 10/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 20s 22ms/step - loss: 0.0016 - val_loss: 0.0016
Epoch 11/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 23s 25ms/step - loss: 0.0016 - val_loss: 0.0016
Epoch 12/100
931/93

In [7]:
# ── W20 데이터 로드 ───────────────────────────────────────────
PREP = 'PREP_DIR'
X_train = np.load(PREP + 'X_train_w20.npy')
X_val   = np.load(PREP + 'X_val_w20.npy')
X_te_tue = np.load(PREP + 'X_te_tue_w20.npy')
X_te_wed = np.load(PREP + 'X_te_wed_w20.npy')
X_te_fri = np.load(PREP + 'X_te_fri_w20.npy')
y_te_tue = np.load(PREP + 'y_te_tue_w20.npy', allow_pickle=True)
y_te_wed = np.load(PREP + 'y_te_wed_w20.npy', allow_pickle=True)
y_te_fri = np.load(PREP + 'y_te_fri_w20.npy', allow_pickle=True)

TIMESTEPS = X_train.shape[1]  # 20
FEATURES  = X_train.shape[2]  # 78
print(f"Train: {X_train.shape} / Val: {X_val.shape}")

Train: (238257, 20, 78) / Val: (26473, 20, 78)


In [8]:
# ── LSTM Autoencoder ──────────────────────────────────────────
from tensorflow.keras.layers import LSTM

print("\n========== EXP4: LSTM ==========")
inp     = Input(shape=(TIMESTEPS, FEATURES))
encoded = LSTM(64, activation='relu', return_sequences=False)(inp)
repeated= RepeatVector(TIMESTEPS)(encoded)
decoded = LSTM(64, activation='relu', return_sequences=True)(repeated)
out     = TimeDistributed(Dense(FEATURES))(decoded)
lstm_model = Model(inp, out)
lstm_model.compile(optimizer='adam', loss='mse')
lstm_model.summary()

start = time.time()
lstm_hist = lstm_model.fit(
    X_train, X_train,
    epochs=100, batch_size=256,
    validation_data=(X_val, X_val),
    callbacks=[es], shuffle=True, verbose=1
)
lstm_time = (time.time() - start) / len(lstm_hist.history['loss'])
lstm_threshold = get_threshold(lstm_model, X_val)
print(f"LSTM 임계값: {lstm_threshold:.6f} / 에폭당 평균: {lstm_time:.2f}s")
lstm_results = evaluate(lstm_model, lstm_threshold, 'LSTM')

lstm_model.save('OUTPUT_DIRmodels/lstm_model.keras')
np.save('OUTPUT_DIRmodels/lstm_threshold.npy', lstm_threshold)

del lstm_model
gc.collect()
print("LSTM 완료 + 저장!")


========== EXP4: LSTM ==========


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 20, 78)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        36,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_1 (RepeatVector)  │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 20, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 20, 78)         │         5,070 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 74,702 (291.80 KB)

 Trainable params: 74,702 (291.80 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 75s 76ms/step - loss: 0.0177 - val_loss: 0.0117
Epoch 2/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 71s 76ms/step - loss: 0.0115 - val_loss: 0.0108
Epoch 3/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 72s 77ms/step - loss: 0.0107 - val_loss: 0.0102
Epoch 4/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 72s 77ms/step - loss: 0.0101 - val_loss: 0.0096
Epoch 5/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 71s 76ms/step - loss: 0.0095 - val_loss: 0.0093
Epoch 6/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 73s 78ms/step - loss: 0.0092 - val_loss: 0.0089
Epoch 7/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 75s 81ms/step - loss: 0.0089 - val_loss: 0.0087
Epoch 8/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 72s 77ms/step - loss: 0.0086 - val_loss: 0.0085
Epoch 9/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 71s 76ms/step - loss: 0.0084 - val_loss: 0.0081
Epoch 10/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 71s 76ms/step - loss: 0.0081 - val_loss: 0.0079
Epoch 11/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 71s 76ms/step - loss: 0.0078 - val_loss: 0.0077
Epoch 12/100
931/93

In [9]:
# ── Bi-LSTM Autoencoder ───────────────────────────────────────
from tensorflow.keras.layers import Bidirectional

print("\n========== EXP5: Bi-LSTM ==========")
inp     = Input(shape=(TIMESTEPS, FEATURES))
encoded = Bidirectional(LSTM(64, activation='relu', return_sequences=False))(inp)
repeated= RepeatVector(TIMESTEPS)(encoded)
decoded = Bidirectional(LSTM(64, activation='relu', return_sequences=True))(repeated)
out     = TimeDistributed(Dense(FEATURES))(decoded)
bilstm_model = Model(inp, out)
bilstm_model.compile(optimizer='adam', loss='mse')
bilstm_model.summary()

start = time.time()
bilstm_hist = bilstm_model.fit(
    X_train, X_train,
    epochs=100, batch_size=256,
    validation_data=(X_val, X_val),
    callbacks=[es], shuffle=True, verbose=1
)
bilstm_time = (time.time() - start) / len(bilstm_hist.history['loss'])
bilstm_threshold = get_threshold(bilstm_model, X_val)
print(f"Bi-LSTM 임계값: {bilstm_threshold:.6f} / 에폭당 평균: {bilstm_time:.2f}s")
bilstm_results = evaluate(bilstm_model, bilstm_threshold, 'Bi-LSTM')

bilstm_model.save('OUTPUT_DIRmodels/bilstm_model.keras')
np.save('OUTPUT_DIRmodels/bilstm_threshold.npy', bilstm_threshold)

del bilstm_model
gc.collect()
print("Bi-LSTM 완료 + 저장!")


========== EXP5: Bi-LSTM ==========


Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 20, 78)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        73,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_2 (RepeatVector)  │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 20, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 20, 78)         │        10,062 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 182,094 (711.30 KB)

 Trainable params: 182,094 (711.30 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 124s 126ms/step - loss: 0.0161 - val_loss: 0.0104
Epoch 2/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 115s 123ms/step - loss: 0.0101 - val_loss: 0.0091
Epoch 3/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 118s 127ms/step - loss: 0.0089 - val_loss: 0.0082
Epoch 4/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 113s 121ms/step - loss: 0.0081 - val_loss: 0.0076
Epoch 5/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 116s 125ms/step - loss: 0.0075 - val_loss: 0.0070
Epoch 6/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 115s 124ms/step - loss: 0.0069 - val_loss: 0.0066
Epoch 7/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 117s 125ms/step - loss: 0.0064 - val_loss: 0.0059
Epoch 8/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 114s 123ms/step - loss: 0.0058 - val_loss: 0.0054
Epoch 9/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 115s 124ms/step - loss: 0.0053 - val_loss: 0.0049
Epoch 10/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 115s 123ms/step - loss: 0.0048 - val_loss: 0.0043
Epoch 11/100
931/931 ━━━━━━━━━━━━━━━━━━━━ 112s 120ms/step - loss: 0.0042 - val_loss: 0.00

In [10]:
# ── MLP 튜닝 ─────────────────────────────────────────────────
# - 시간 순서 무시 (Flatten으로 펼침)
# - 단순 패턴 복원에 강함
# - 레이어 깊이에 민감
# - Dropout 없으면 과적합 위험

# 튜닝 항목:
# 1. units: 모델 복잡도 (64 vs 128)
# 2. percentile: 임계값 엄격도 (90 vs 95)
# 3. dropout: 과적합 방지 (있음 vs 없음)
#
# TUN1: units=128, p=95, dropout=X → 용량 증가 효과 확인
# TUN2: units=64,  p=90, dropout=X → 임계값 조정 효과 확인
# TUN3: units=128, p=90, dropout=X → 용량+임계값 복합 효과 확인
# TUN4: units=64,  p=95, dropout=0.1 → 과적합 방지 효과 확인
#       MLP는 시간 순서 무시하고 단순 복원 → 과적합 가능성 존재
#       Dropout으로 일반화 성능 향상 기대

from tensorflow.keras.layers import Dropout

# W5 데이터 로드
PREP = 'PREP_DIR'
X_train = np.load(PREP + 'X_train_w5.npy')
X_val   = np.load(PREP + 'X_val_w5.npy')
X_te_tue = np.load(PREP + 'X_te_tue_w5.npy')
X_te_wed = np.load(PREP + 'X_te_wed_w5.npy')
X_te_fri = np.load(PREP + 'X_te_fri_w5.npy')
y_te_tue = np.load(PREP + 'y_te_tue_w5.npy', allow_pickle=True)
y_te_wed = np.load(PREP + 'y_te_wed_w5.npy', allow_pickle=True)
y_te_fri = np.load(PREP + 'y_te_fri_w5.npy', allow_pickle=True)

TIMESTEPS = X_train.shape[1]
FEATURES  = X_train.shape[2]

tuning_configs = [
    # (units, percentile, dropout_rate, 설명)
    (128, 95, 0.0,  'TUN1_u128_p95_nodrop'),   # 용량 증가
    (64,  90, 0.0,  'TUN2_u64_p90_nodrop'),    # 임계값 조정
    (128, 90, 0.0,  'TUN3_u128_p90_nodrop'),   # 용량+임계값
    (64,  95, 0.1,  'TUN4_u64_p95_drop01'),    # 과적합 방지
]

tuning_results = {}

for units, percentile, dropout_rate, name in tuning_configs:
    print(f"\n========== {name} ==========")

    inp = Input(shape=(TIMESTEPS, FEATURES))
    x   = Flatten()(inp)
    x   = Dense(units*2, activation='relu')(x)
    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)  # 과적합 방지
    x   = Dense(units, activation='relu')(x)
    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)
    x   = Dense(units*2, activation='relu')(x)
    out = Dense(TIMESTEPS * FEATURES, activation='sigmoid')(x)
    out = Reshape((TIMESTEPS, FEATURES))(out)

    model = Model(inp, out)
    model.compile(optimizer='adam', loss='mse')

    hist = model.fit(
        X_train, X_train,
        epochs=100, batch_size=256,
        validation_data=(X_val, X_val),
        callbacks=[es], shuffle=True, verbose=0
    )

    threshold = get_threshold(model, X_val, percentile)
    results   = evaluate(model, threshold, name)

    tuning_results[name] = {
        'units': units,
        'percentile': percentile,
        'dropout': dropout_rate,
        'threshold': threshold,
        'epochs': len(hist.history['loss']),
        'final_loss': hist.history['loss'][-1],
        'results': results
    }

    model.save(f'OUTPUT_DIRmodels/{name}.keras')
    np.save(f'OUTPUT_DIRmodels/{name}_threshold.npy', threshold)

    del model
    gc.collect()



========== TUN1_u128_p95_nodrop ==========
  [TUN1_u128_p95_nodrop] BruteForce → Recall: 0.064  Precision: 0.040  F1: 0.049
  [TUN1_u128_p95_nodrop] DoS → Recall: 0.707  Precision: 0.764  F1: 0.735
  [TUN1_u128_p95_nodrop] DDoS → Recall: 0.824  Precision: 0.755  F1: 0.788

========== TUN2_u64_p90_nodrop ==========
  [TUN2_u64_p90_nodrop] BruteForce → Recall: 0.128  Precision: 0.040  F1: 0.061
  [TUN2_u64_p90_nodrop] DoS → Recall: 0.716  Precision: 0.690  F1: 0.703
  [TUN2_u64_p90_nodrop] DDoS → Recall: 0.836  Precision: 0.739  F1: 0.784

========== TUN3_u128_p90_nodrop ==========
  [TUN3_u128_p90_nodrop] BruteForce → Recall: 0.117  Precision: 0.035  F1: 0.054
  [TUN3_u128_p90_nodrop] DoS → Recall: 0.650  Precision: 0.749  F1: 0.696
  [TUN3_u128_p90_nodrop] DDoS → Recall: 0.686  Precision: 0.798  F1: 0.738

========== TUN4_u64_p95_drop01 ==========
  [TUN4_u64_p95_drop01] BruteForce → Recall: 0.069  Precision: 0.035  F1: 0.047
  [TUN4_u64_p95_drop01] DoS → Recall: 0.706  Precision: 0.7

In [11]:
print("\n========== MLP 튜닝 결과 비교 ==========")
print(f"{'모델':<25} {'평균Recall':>10} {'BruteForce':>12} {'DoS':>8} {'DDoS':>8} {'Threshold':>12}")
print("-" * 80)

# 베이스 결과 먼저 출력
print(f"{'BASE_u64_p95_nodrop':<25} {'0.534':>10} {'0.061':>12} {'0.710':>8} {'0.832':>8} {'0.000202':>12}")

# 튜닝 결과 출력
for name, r in tuning_results.items():
    avg = np.mean([r['results'][k]['recall'] for k in r['results']])
    bf  = r['results']['BruteForce']['recall']
    dos = r['results']['DoS']['recall']
    dds = r['results']['DDoS']['recall']
    print(f"{name:<25} {avg:>10.3f} {bf:>12.3f} {dos:>8.3f} {dds:>8.3f} {r['threshold']:>12.6f}")

print("\nMLP 튜닝 완료!")


========== MLP 튜닝 결과 비교 ==========
모델                          평균Recall   BruteForce      DoS     DDoS    Threshold
--------------------------------------------------------------------------------
BASE_u64_p95_nodrop            0.534        0.061    0.710    0.832     0.000202
TUN1_u128_p95_nodrop           0.531        0.064    0.707    0.824     0.000146
TUN2_u64_p90_nodrop            0.560        0.128    0.716    0.836     0.000150
TUN3_u128_p90_nodrop           0.484        0.117    0.650    0.686     0.021899
TUN4_u64_p95_drop01            0.525        0.069    0.706    0.801     0.001205

MLP 튜닝 완료!
